# Agent 4 — Video Searching Agent (Public Platforms)

> Discover videos across YouTube, TikTok, Instagram, and X using
> natural-language queries. The agentic loop runs server-side via
> the managed `POST /queries/stream` SSE endpoint.

## What's different about this agent

The other agents in this collection compose Visual Search + Visual
Intelligence themselves. This one calls a single managed endpoint
(`/queries/stream`) and reads typed Server-Sent Events as the
server-side agent does its work:

| Event | When it fires |
|---|---|
| `started` | Session begins |
| `progress` | Agent advances one step (out of N) |
| `tool_call` | Agent invokes a tool (e.g. `tiktok_search`) |
| `tool_result` | Tool returns |
| `error` | Recoverable error |
| `complete` | Final structured response — the stream closes after this |

## Endpoints exercised

| Step | Endpoint |
|---|---|
| Submit & stream | `POST /queries/stream` (SSE) |


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


## Helper — minimal SSE consumer

SSE (Server-Sent Events) is a one-way streaming protocol where each
message has an `event:` name and a `data:` payload. The full spec is
small; we only need ~15 lines.


In [ ]:
def iter_sse(resp):
    """Yield (event_name, parsed_data) tuples from an SSE response.

    Format:
        event: <name>
        data: <json>
        <blank line>

    Lines starting with ':' are comments. Data may span multiple
    'data:' lines; they're joined with newlines.
    """
    event_name = "message"
    data_lines = []
    for raw in resp.iter_lines(decode_unicode=True):
        if raw is None:
            continue
        if raw == "":
            # End of one event — emit it.
            if data_lines:
                payload = "\n".join(data_lines)
                try:
                    parsed = json.loads(payload)
                except (TypeError, ValueError):
                    parsed = payload  # not JSON; return raw string
                yield event_name, parsed
            event_name = "message"
            data_lines = []
            continue
        if raw.startswith(":"):
            continue  # SSE comment
        if raw.startswith("event:"):
            event_name = raw[len("event:"):].strip()
        elif raw.startswith("data:"):
            data_lines.append(raw[len("data:"):].lstrip())


## Step 1 — submit the query and stream events

We open the request with `stream=True` so we can read events as they
arrive rather than waiting for the full response. The stream may run
for tens of seconds while the server-side agent searches multiple
platforms.


In [ ]:
body = {
    "query": "Find trending sous vide tutorials on TikTok with clear narration",
    "platforms": ["tiktok"],         # one or more of: youtube, tiktok, instagram, twitter
    "max_results": 3,
    "time_frame": None,              # e.g. "past_24h", "past_week", "past_month"
}

final_result = None
with requests.post(
    f"{VLM_HOST}/queries/stream",
    headers={**HEADERS, "Accept": "text/event-stream"},
    json=body,
    stream=True,
    timeout=300,
) as resp:
    resp.raise_for_status()
    for event_name, data in iter_sse(resp):
        if event_name == "started":
            print(f"started session={data.get('session_id')}")
        elif event_name == "progress":
            print(f"  step {data.get('step')}/{data.get('max_steps')}: {data.get('message')}")
        elif event_name == "tool_call":
            print(f"    -> tool_call: {data.get('tool')}({data.get('arguments')})")
        elif event_name == "tool_result":
            ok = "ok" if data.get("success") else "FAIL"
            print(f"    <- tool_result [{ok}]: {data.get('summary')}")
        elif event_name == "error":
            print(f"  ERROR: {data.get('message')}")
        elif event_name == "complete":
            final_result = data
            refs = (data or {}).get("video_references", []) or []
            print(f"\ncomplete: {len(refs)} videos, confidence={data.get('confidence_score')}")


## Step 2 — inspect the structured result

The `complete` event payload includes the natural-language answer plus
a structured list of video references. Each reference has the platform
URL, creator, engagement stats, and a relevance note explaining why it
matched.


In [ ]:
print("Answer:")
print(final_result.get("answer", "")[:600], "...\n")

print(f"\n{len(final_result.get('video_references', []))} videos found:\n")
for ref in (final_result.get("video_references") or [])[:5]:
    print(f"  • [{ref.get('platform')}] {ref.get('title')[:60]}")
    print(f"    by {ref.get('creator')}  ({ref.get('views', 0):,} views)")
    print(f"    {ref.get('url')}")
    print(f"    why: {ref.get('relevance_note')}\n")


## Where to go next

- **Comparative queries**: ask "How does Creator A's style compare to
  Creator B's?" — the agent will retrieve representative videos from
  each and synthesize a comparison.
- **Trend detection**: filter `time_frame="past_24h"` and run the same
  query daily to catch emerging formats early.
- **Multi-platform**: pass `["youtube", "tiktok", "instagram"]` to
  have the server-side agent merge across platforms in one call.
